# EXAMS (mhardalov/exams) — Data Inspection

Checks: language filtering (Italian/Spanish), choice count distribution,
encoding issues, support paragraph content, and produces a cleaned CSV
if needed.

In [ ]:
from datasets import load_dataset
import pandas as pd
import re

TARGET_LANGUAGES = {'Italian', 'Spanish'}

# Use the multilingual config — has all languages in one split
ds = load_dataset('mhardalov/exams', 'multilingual')
print(ds)

In [ ]:
# Inspect raw structure
ex = ds['full'][0]
print('Top-level keys:', list(ex.keys()))
print()
print('question keys:', list(ex['question'].keys()))
print()
print('choices keys:', list(ex['question']['choices'].keys()))
print()
print('Example:')
print(f"  stem: {ex['question']['stem'][:100]}")
print(f"  labels: {ex['question']['choices']['label']}")
print(f"  texts: {ex['question']['choices']['text']}")
print(f"  para: {ex['question']['choices']['para']}")
print(f"  answerKey: {ex['answerKey']}")
print(f"  info: {ex['info']}")

In [ ]:
# Flatten into dataframe
rows = []
for ex in ds['full']:
    info = ex.get('info', {})
    choices = ex['question']['choices']
    rows.append({
        'id':          ex['id'],
        'language':    info.get('language', ''),
        'subject':     info.get('subject', ''),
        'grade':       info.get('grade', ''),
        'stem':        ex['question']['stem'],
        'labels':      choices['label'],
        'texts':       choices['text'],
        'paras':       choices['para'],
        'n_choices':   len(choices['label']),
        'answer_key':  ex['answerKey'],
    })

df = pd.DataFrame(rows)
print(df.shape)
df.head(2)

In [ ]:
# Language distribution
print(df['language'].value_counts().to_string())

In [ ]:
# Filter to target languages
df_target = df[df['language'].isin(TARGET_LANGUAGES)].copy()
print(f'Rows for Italian/Spanish: {len(df_target)}')
print(df_target['language'].value_counts())

In [ ]:
# Choice count distribution — may be 3, 4 or 5
print('Choice count distribution:')
print(df_target.groupby(['language', 'n_choices']).size().to_string())

In [ ]:
# Check answerKey values
print('Answer key values:')
print(df_target['answer_key'].value_counts())

# Flag answer keys not in labels
bad_key = df_target[df_target.apply(
    lambda r: r['answer_key'] not in r['labels'], axis=1
)]
print(f'\nRows where answer_key not in labels: {len(bad_key)}')
if len(bad_key):
    display(bad_key[['id', 'language', 'stem', 'labels', 'answer_key']].head(10))

In [ ]:
# Check support paragraphs (para field) — are they ever non-empty?
def has_para(paras):
    return any(p and str(p).strip() for p in paras)

df_target['has_para'] = df_target['paras'].apply(has_para)
print(f'Rows with non-empty support paragraph: {df_target["has_para"].sum()}')
print(df_target.groupby(['language', 'has_para']).size())

# Show an example with a para
with_para = df_target[df_target['has_para']]
if len(with_para):
    ex = with_para.iloc[0]
    print(f'\nExample with para:')
    print(f'  stem: {ex["stem"][:100]}')
    print(f'  paras: {ex["paras"]}')

In [ ]:
# Encoding corruption check
corruption = re.compile(r'[\ufffd\x00-\x08\x0b\x0c\x0e-\x1f]')

def has_corruption(row):
    fields = [row['stem']] + list(row['texts'])
    return any(corruption.search(str(f)) for f in fields)

df_target['has_corruption'] = df_target.apply(has_corruption, axis=1)
print(f'Rows with encoding corruption: {df_target["has_corruption"].sum()}')
if df_target['has_corruption'].any():
    display(df_target[df_target['has_corruption']][['language', 'stem', 'texts']].head(5))

In [ ]:
# Subject distribution per language
for lang in TARGET_LANGUAGES:
    sub = df_target[df_target['language'] == lang]
    print(f'\n=== {lang} ({len(sub)} rows) ===')
    print(sub['subject'].value_counts().to_string())

In [ ]:
# Spot-check formatted examples
for lang in TARGET_LANGUAGES:
    sub = df_target[df_target['language'] == lang]
    row = sub.iloc[0]
    print(f'=== {lang} ===')
    print(f'Q: {row["stem"]}')
    for label, text in zip(row['labels'], row['texts']):
        print(f'  {label}) {text}')
    print(f'Answer: {row["answer_key"]}')
    print()

In [ ]:
# Cleaning decision
# Keep rows where:
#   - language in target
#   - answer_key in labels
#   - no encoding corruption
#   - stem is non-empty
#   - at least 2 choices (we accept variable counts)

clean = df_target[
    df_target.apply(lambda r: r['answer_key'] in r['labels'], axis=1) &
    ~df_target['has_corruption'] &
    (df_target['stem'].str.strip() != '') &
    (df_target['n_choices'] >= 2)
].copy()

rejected = df_target[~df_target.index.isin(clean.index)].copy()

print(f'Clean:    {len(clean)}')
print(f'Rejected: {len(rejected)}')
print()
print(clean.groupby(['language', 'n_choices']).size())

In [ ]:
# Save cleaned version
import json
from pathlib import Path

out_dir = Path('multilingual/data/processed/exams')
out_dir.mkdir(parents=True, exist_ok=True)

# Convert list columns to JSON strings for CSV compatibility
for col in ['labels', 'texts', 'paras']:
    clean[col] = clean[col].apply(json.dumps)
    rejected[col] = rejected[col].apply(json.dumps)

clean.to_csv(out_dir / 'exams_clean_all.csv', index=False)
rejected.to_csv(out_dir / 'exams_rejected_all.csv', index=False)

for lang in TARGET_LANGUAGES:
    lang_clean = clean[clean['language'] == lang]
    lang_clean.to_csv(out_dir / f'exams_clean_{lang.lower()}.csv', index=False)
    print(f'Saved {lang}: {len(lang_clean)} rows')

print(f'Saved clean:    {out_dir / "exams_clean_all.csv"}')
print(f'Saved rejected: {out_dir / "exams_rejected_all.csv"}')

In [ ]:
# Summary
print('=== SUMMARY ===')
print(f'Total multilingual rows:         {len(df)}')
print(f'Italian + Spanish rows:          {len(df_target)}')
print(f'Clean after filtering:           {len(clean)}')
print(f'Rejected:                        {len(rejected)}')
print()
print('Choice count in clean data:')
print(clean.groupby(['language', 'n_choices']).size().to_string())
print()
print('Note: variable choice counts (3-5) are acceptable.')
print('The loader should use all labels present, not assume A/B/C/D.')